In [ ]:
%%capture
import os
from pathlib import Path

import pandas as pd
from dj_notebook import activate

env_file = os.environ["INTECOMM_ENV"]
analysis_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
reports_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
plus = activate(dotenv_file=env_file)
pd.set_option('future.no_silent_downcasting', True)

In [ ]:
# pip install folium selenium

from intecomm_group.maps.get_maps import format_df
from selenium import webdriver
from tempfile import mktemp
import PIL
import folium
import io
import time

In [ ]:
country_locations = {"uganda": [0.4044, 32.4594], "tanzania": [-6.7039, 39.0406]}

In [ ]:
def export_map_to_png(folium_map: folium.Map, filename: Path):
    """
    Exports a Folium map to a PNG image using Selenium.
    """
    options = webdriver.firefox.options.Options()
    options.add_argument("--headless")
    options.add_argument("--hide-scrollbars")
    options.add_argument("--force-device-scale-factor=2")
    driver = webdriver.Firefox(options=options)

    img_data = folium_map._to_png(delay=5, driver=driver, size=[750,490])
    img = PIL.Image.open(io.BytesIO(img_data))
    img.save(analysis_folder / filename)
    print(f"Map successfully exported to {filename}")


def export_map_to_png2(folium_map: folium.Map, filename: Path):
    """
    Exports a Folium map to a PNG image using Selenium.
    """
    # Create a temporary HTML file
    temp_html_file = Path(mktemp(suffix=".html"))
    folium_map.save(temp_html_file)

    # Configure Chrome options for headless mode
    chrome_options = webdriver.chrome.options.Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--hide-scrollbars")
    chrome_options.add_argument("--force-device-scale-factor=2")

    driver = webdriver.Chrome(options=chrome_options)
    driver.set_window_size(750, 490)

    try:
        driver.get(f"file://{temp_html_file}")
        time.sleep(5)  # Give the map time to render fully
        driver.save_screenshot(filename)
        print(f"Map successfully exported to {filename}")
    finally:
        driver.quit()
        # Clean up the temporary HTML file
        if temp_html_file.exists() and temp_html_file.is_file():
            temp_html_file.unlink()

In [ ]:
def get_coordinates_df(path) -> pd.DataFrame:
    df_ug = pd.read_csv(path / "Intecomm-coordinates(Uganda).csv")
    df_ug["country"] = "uganda"
    df_ug = format_df(df_ug)
    df_ug = df_ug[["name", "lat", "lon", "location_type", "country"]]
    df_ug["facility"] = pd.NA
    df_ug.loc[df_ug["location_type"] == "facility", "facility"] = df_ug.loc[
        df_ug["location_type"] == "facility"
    ]["name"]
    df_ug["facility"] = df_ug.facility.ffill()

    df_tz = pd.read_csv(path / "Intecomm-coordinates(Tanzania1).csv")
    df_tz["country"] = "tanzania"
    df_tz["community_groups"] = pd.NA
    df_tz = format_df(df_tz)
    df_tz = df_tz[["name", "lat", "lon", "location_type", "country"]]
    df_tz["facility"] = pd.NA
    df_tz.loc[df_tz["location_type"] == "facility", "facility"] = df_tz.loc[
        df_tz["location_type"] == "facility"
    ]["name"]
    df_tz["facility"] = df_tz.facility.ffill()
    return pd.concat([df_tz, df_ug])


In [ ]:
def get_map(country:str, tiles:str|None=None, attr:str|None=None) -> folium.Map:
    df_coordinates = get_coordinates_df(analysis_folder)
    map = folium.Map(
        location=country_locations[country],
        zoom_start=10,
        tiles=tiles or "CartoDB Voyager",
        zoom_control=False,
        attr=attr,
        width="750px",
        height="490px",
    )

    for _, row in df_coordinates.query("location_type=='community' and country==@country").iterrows():
        folium.Circle(
            radius=500,
            location=[row["lat"], row["lon"]],
            # popup=f'{row["name"]} [{row["facility"]}]',
            fill=False,
            color="orange",
            fill_opacity=1,
        ).add_to(map)
    for _, row in df_coordinates.query("location_type=='facility' and country==@country").iterrows():
            folium.Circle(
                location=[row["lat"], row["lon"]],
                radius=5000,  # 5 km in meters
                color="gray",
                fill=True,
                fill_opacity=0.1,
                weight=0,
            ).add_to(map)
            folium.Circle(
                radius=600,
                location=[row["lat"], row["lon"]],
                popup=row["name"],
                fill=True,
                color="green",
                fill_opacity=1,
            ).add_to(map)
    return map

In [ ]:
def get_title_html(country:str):
    return f'''<div style="position: fixed;
                top: 10px; left: 10px; width: 150px; height: 50px;
                z-index:9999;font-face:Fira Sans;opacity:0.8;
                background-color:transparent;">
    <h3 align="left" style="margin-top: 5px;">{country.upper()}</h3>
    </div>
    '''

In [ ]:
def get_legend_html():
    color_map = {
        'Community-care': 'orange',
        'Facility-care <sup>*</sup>': 'green',
    }
    legend_html = '''
         <div style="position: fixed;
                     bottom: 50px; left: 10px; width: 150px; height: 100px;
                     border:1px solid grey; z-index:9999; font-size:14px;font-face:Fira Sans;
                     background-color:white; opacity:0.9;">
           <div style="padding: 10px;">
             <b>CLINIC TYPES</b> <br>
    '''
    for category, color in color_map.items():
        legend_html += f'''
              <div style="display: flex; align-items: center; margin-bottom: 5px;">
                <i style="background: {color}; width: 18px; height: 18px; margin-right: 5px;font-face:Fira Sans;"></i>
                <span>{category}</span>
              </div>
        '''
    legend_html += '''
              <div style="display: flex; align-items: center; margin-bottom: 5px; font-size:10px;">
                <span>* Showing 5km radius</span>
              </div>'''

    legend_html += '''
           </div>
        </div>
    '''
    return legend_html

In [ ]:
for country in ["uganda", "tanzania"]:
    country_map = get_map(country, tiles="CartoDB Voyager", attr="CartoDB")
    # add legend tp map object
    country_map.get_root().html.add_child(folium.Element(get_legend_html()))
    country_map.get_root().html.add_child(folium.Element(get_title_html(country)))

    # to html
    country_map.save(analysis_folder / f"intecomm_{country}_map.html")
    # to PGN
    export_map_to_png(country_map, analysis_folder / f"intecomm_{country}_map.png")
